In [ ]:
%%writefile data_processing.py
import pandas as pd
import numpy as np
import random
import gc
import time
import shutil
import re
import os
from tqdm import tqdm
import glob
# from unidecode import unidecode
from parameter import Parameter
from utils import KFold

parameter = Parameter()


def get_data(seed=27, mode=0):
    if os.path.exists('../input/ai4codetrainpicklefile/train_df.pkl'):
        train_df = pd.read_pickle('../input/ai4codetrainpicklefile/train_df.pkl')
    else:
        train_df = read_json_data(mode='train')
        train_orders = pd.read_csv(parameter.data_dir + 'train_orders.csv')
        train_ancestors = pd.read_csv(parameter.data_dir + 'train_ancestors.csv')

        train_orders['cell_id'] = train_orders['cell_order'].str.split()
        train_orders = train_orders.explode(column='cell_id')
        train_orders['rank'] = train_orders.groupby(by='id').cumcount() + 1
        # train_orders['flag'] = range(len(train_orders))
        # train_orders['rank'] = train_orders.groupby(by=['id'])['flag'].rank(ascending=True, method='first').astype(int)
        # del train_orders['flag'], train_orders['cell_order']
        del train_orders['cell_order']
        print(train_orders)
        # train_df = preprocess_features(train_df)
        train_df = train_df.merge(train_orders, on=['id', 'cell_id'], how='left')
        train_df = train_df.merge(train_ancestors[['id', 'ancestor_id']], on=['id'], how='left')
        train_df.to_pickle('train_df.pkl')

    train_df = KFold(seed, parameter.k_folds).group_split(train_df, group_col='ancestor_id')
    # train_df = preprocess_features(train_df)
    # train_df['source_length'] = train_df['source'].apply(len)
    # train_df['id_length'] = train_df.groupby(by=['id'])['source_length'].transform('sum')
    train_df = preprocess_df(train_df)
    train_df = pd.concat(
        [train_df[train_df['cell_type'] == 0], train_df[train_df['cell_type'] == 1].sample(frac=1.0)]).reset_index(
        drop=True)
    train_df['rank2'] = (train_df.groupby(by=['id', 'cell_type']).cumcount() + 1) / \
                        train_df.groupby(by=['id', 'cell_type'])['cell_id'].transform('count')
    train_df.loc[train_df['cell_type'] == 1, 'rank2'] = -1
    code_df_valid = train_df[train_df['cell_type'] == 0][['id', 'cell_id', 'rank2']].copy()
    
    for col in ['cell_count','markdown_count', 'code_count']:
        train_df[col] = (train_df[col] - train_df[col].mean())/ train_df[col].std()
        train_df[col] = np.clip(train_df[col].fillna(0.0), -3, 3)

    train_df = get_truncated_df(train_df, cell_count=parameter.cell_count)
#     train_df['flag'] = train_df['cell_type'].apply(lambda x:np.sum(x))
#     train_df = train_df[train_df['flag']>0]
#     del train_df['flag']
    print(train_df)
    print(train_df.shape)
    return train_df, code_df_valid


def read_json_data(mode='train'):
    paths_train = sorted(list(glob.glob(parameter.data_dir + '{}/*.json'.format(mode))))  # [:100]
    res = pd.concat([
        pd.read_json(path, dtype={'cell_type': 'category', 'source': 'str'}).assign(
            id=path.split('/')[-1].split('.')[0]).rename_axis('cell_id')
        for path in tqdm(paths_train)]).reset_index(drop=False)
    res = res[['id', 'cell_id', 'cell_type', 'source']]
    return res


def preprocess_df(df):
    df['cell_count'] = df.groupby(by=['id'])['cell_id'].transform('count')
    # df['source'] = df['cell_type'] + ' ' + df['source']
    df['cell_type'] = df['cell_type'].map({'code': 0, 'markdown': 1}).fillna(0).astype(int)
    # df.loc[df['cell_type']==0, 'source'] = df.loc[df['cell_type']==0, 'rank'] + ' ' + df.loc[df['cell_type']==0, 'source']
    df['markdown_count'] = df.groupby(by=['id'])['cell_type'].transform('sum')
    df['code_count'] = df['cell_count'] - df['markdown_count']
    df['rank'] = df['rank'] / df['cell_count']
    df['source'] = df['source'].apply(lambda x: x.lower().strip())
    df['source'] = df['source'].apply(lambda x:preprocess_text(x))
    # df['source'] = df['source'].replace("\\n", "\n")
    # df['source'] = df['source'].str.replace("\n", "")
    df['source'] = df['source'].str.replace("[SEP]", "")
    df['source'] = df['source'].str.replace("[CLS]", "")

    # df['source'] = df['source'].replace("#", "")
    # df['source'] = df['source'].apply(lambda x: unidecode(x))
    df['source'] = df['source'].apply(lambda x: re.sub(' +', ' ', x))
    return df

# from https://www.kaggle.com/code/ilyaryabov/fastttext-sorting-with-cosine-distance-algo
import re
from nltk.stem import WordNetLemmatizer

stemmer = WordNetLemmatizer()

def preprocess_text(document):
        # Remove all the special characters
        document = re.sub(r'\W', ' ', str(document))
        document = document.replace('_',' ')

        # remove all single characters
        document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)

#         # Remove single characters from the start
#         document = re.sub(r'\^[a-zA-Z]\s+', ' ', document)

        # Substituting multiple spaces with single space
        document = re.sub(r'\s+', ' ', document, flags=re.I)

#         # Removing prefixed 'b'
#         document = re.sub(r'^b\s+', '', document)

        # Converting to Lowercase
        document = document.lower()
        #return document

#         # Lemmatization
#         tokens = document.split()
#         tokens = [stemmer.lemmatize(word) for word in tokens]
#         # tokens = [word for word in tokens if len(word) > 3]

#         preprocessed_text = ' '.join(tokens)
        return document

def get_truncated_df(df, cell_count=128, id_col='id2', group_col='id', max_random_cnt=100, expand_ratio=5):
    tmp1 = df[df['cell_count'] <= cell_count].reset_index(drop=True)
    tmp1.loc[:, id_col] = 1
    tmp2 = df[df['cell_count'] > cell_count].reset_index(drop=True)
    # print(tmp1.shape,tmp2.shape)
    res = [tmp1]
    for _, df_g in tmp2.groupby(by=group_col):
        # print(df_g.columns)
        df_g = df_g.sample(frac=1.0).reset_index(drop=True)
        step = min(cell_count // 2, len(df_g) - cell_count)
        step = max(step, 1)
        id_col_count = 1
        for i in range(0, len(df_g), step):
            res_tmp = df_g.iloc[i:i + cell_count]  # .copy()
            if len(res_tmp) != cell_count:
                res_tmp = df_g.iloc[-cell_count:]
            # if len(res_tmp) == cell_count:
            res_tmp.loc[:, id_col] = id_col_count
            id_col_count += 1
            res.append(res_tmp)
            if i + cell_count >= len(df_g):
                break

        if len(df_g) // cell_count > 1.3:
            random_cnt = int(len(df_g) // cell_count * expand_ratio)
            random_cnt = min(random_cnt, max_random_cnt)  # todo

            for i in range(random_cnt):
                res_tmp = df_g.sample(n=cell_count).reset_index(drop=True)
                res_tmp.loc[:, id_col] = id_col_count
                id_col_count += 1
                res.append(res_tmp)

    res = pd.concat(res).reset_index(drop=True)
    res = res.sort_values(by=['id', id_col, 'cell_type', 'rank2'], ascending=True)
    res = res.groupby(by=['id', id_col, 'fold_flag', 'cell_count', 'markdown_count', 'code_count'], as_index=False, sort=False)[
        ['cell_id', 'cell_type', 'source', 'rank', 'rank2']].agg(list)
    return res


def get_truncated_df2(df, cell_count=128, id_col='id2', group_col='id'):
    res = []
    for _, df_g in df.groupby(by=group_col):
        # print(df_g.columns)
        df_g = df_g.reset_index(drop=True)
        step = cell_count
        step = max(step, 1)
        id_col_count = 1
        for i in range(0, len(df_g), step):
            res_tmp = df_g.iloc[i:i + cell_count]
            if len(res_tmp) >0:
                res_tmp.loc[:, id_col] = id_col_count
                id_col_count += 1
                res.append(res_tmp)
                if i + cell_count >= len(df_g):
                    break
    res = pd.concat(res).reset_index(drop=True)
    res = res.sort_values(by=['id', id_col, 'cell_type', 'rank2'], ascending=True)
    res = res.groupby(by=['id', id_col], as_index=False, sort=False)[['cell_id', 'cell_type', 'source']].agg(list)
    return res

In [ ]:
%%writefile dataset.py
import pandas as pd
import torch
import random
from torch.utils.data.dataset import Dataset
from torch.utils.data.sampler import Sampler
import os
# from utils import create_label
import numpy as np


class MarkdownDataset(Dataset):

    def __init__(self, meta_data: pd.DataFrame, tokenizer, fold: int = -1, mode='train', parameter=None):
        self.meta_data = meta_data.copy()
        self.meta_data.reset_index(drop=True, inplace=True)
        if mode == 'train':
            self.meta_data = self.meta_data[self.meta_data['fold_flag'] != fold].copy()
            self.meta_data = self.meta_data.iloc[:60000]
        elif mode == 'valid':
            self.meta_data = self.meta_data[self.meta_data['fold_flag'] == fold].copy()
            self.meta_data = self.meta_data[self.meta_data['id'].isin(self.meta_data['id'].values[:1000])]
        elif mode == 'test':
            pass
        else:
            raise ValueError(mode)
        self.meta_data.reset_index(drop=True, inplace=True)
        if tokenizer.sep_token != '[SEP]':
            self.meta_data['source'] = self.meta_data['source'].apply(
                lambda x: [
                    y.replace(tokenizer.sep_token, '').replace(tokenizer.cls_token, '').replace(tokenizer.pad_token, '')
                    for y in x])
        self.parameter = parameter
        self.seq_length = parameter.seq_length
        self.source = self.meta_data['source'].values
        self.cell_type = self.meta_data['cell_type'].values
        # self.cell_id = self.meta_data['cell_id'].values
        self.rank = self.meta_data['rank'].values
        # self.dense_features = self.meta_data[['cell_count','markdown_count', 'code_count']].values
        self.mode = mode
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        source = self.source[index]
        cell_type = self.cell_type[index]
        rank = self.rank[index]
        # dense_features = 1#self.dense_features[index]
#         if self.mode == 'train':
#             range_tmp1 = [ i for i in range(len(cell_type)) if cell_type[i]==0]
#             range_tmp2 = [ i for i in range(len(cell_type)) if cell_type[i]==1]
#             np.random.shuffle(range_tmp2)
#             source = [source[i] for i in range_tmp1 + range_tmp2]
#             rank = [rank[i] for i in range_tmp1 + range_tmp2]
#             rank2 = [rank2[i] for i in range_tmp1 + range_tmp2]

        cell_inputs = self.tokenizer.batch_encode_plus(
            source,
            add_special_tokens=False,
            max_length=self.parameter.cell_max_length,
            # padding="max_length",
            return_attention_mask=False,
            truncation=True,
        )
        seq, seq_mask, target_mask, target = self.max_length_rule_base(cell_inputs['input_ids'],
                                                                                       cell_type, rank)
        # print(seq, seq_mask, dense_features, target_mask, target)
        # if self.mode == 'train':
        #     attention_mask, target = self.random_mask(attention_mask, target)
        # print(encoded)
        # print(target)
        return seq, seq_mask, target_mask, target
        # return encoded['input_ids'][0], encoded['attention_mask'][0], np.array(target, dtype=np.float32)

    def __len__(self):
        return len(self.meta_data)

    def max_length_rule_base(self, cell_inputs, cell_type, rank):
        init_length = [len(x) for x in cell_inputs]
        total_max_length = self.seq_length - len(init_length)
        min_length = total_max_length // len(init_length)
        cell_length = self.search_length(init_length, min_length, total_max_length, len(init_length))
        # print(init_code_length,code_length)

        seq = []
        for i in range(len(cell_length)):
            if cell_type[i] == 0:
                seq.append(self.tokenizer.cls_token_id)
            else:
                seq.append(self.tokenizer.sep_token_id)

            if cell_length[i] > 0:
                seq.extend(cell_inputs[i][:cell_length[i]])

        # print(len(seq),'1111', np.sum(init_length),np.sum(cell_length))
#         if len(seq) < self.seq_length:
#             seq_mask = [1] * len(seq) + [0] * (self.seq_length - len(seq))
#             seq = seq + [self.tokenizer.pad_token_id] * (self.seq_length - len(seq))
#         else:
#             seq_mask = [1] * self.seq_length
#             seq = seq[:self.seq_length]
        seq, seq_mask = np.array(seq, dtype=np.int), np.array(seq_mask, dtype=np.int)
        target_mask = np.where((seq == self.tokenizer.cls_token_id) | (seq == self.tokenizer.sep_token_id), 1, 0)  # todo
        target = np.zeros(len(seq), dtype=np.float32)
        tmp = np.where((seq == self.tokenizer.cls_token_id) | (seq == self.tokenizer.sep_token_id))
        target[tmp] = rank
        sample_weight = np.zeros(len(seq), dtype=np.float32)
        sample_weight = np.where(seq == self.tokenizer.cls_token_id, 0.33, sample_weight)
        sample_weight = np.where(seq == self.tokenizer.sep_token_id, 1.0, sample_weight)
#         dense_features = np.zeros(self.seq_length, dtype=np.float32)
#         dense_features[tmp] = rank2
        return seq, seq_mask, target_mask, target, sample_weight

    @staticmethod
    def search_length(init_length, min_length, total_max_length, cell_count, step=4, max_search_count=50):
        if np.sum(init_length) <= total_max_length:
            return init_length

        res = [min(init_length[i], min_length) for i in range(cell_count)]
        for s_i in range(max_search_count):
            tmp = [min(init_length[i], res[i] + step) for i in range(cell_count)]
            if np.sum(tmp) < total_max_length:
                res = tmp
            else:
                break
        for s_i in range(cell_count):
            tmp = [i for i in res]
            tmp[s_i] = min(init_length[s_i], res[s_i] + step)
            if np.sum(tmp) < total_max_length:
                res = tmp
            else:
                break
        return res
    

class MarkdownDatasetV2(Dataset):

    def __init__(self, meta_data: pd.DataFrame, tokenizer, parameter=None, max_length=4096):
        self.meta_data = meta_data.copy()
        self.meta_data.reset_index(drop=True, inplace=True)
        if tokenizer.sep_token != '[SEP]':
            self.meta_data['source'] = self.meta_data['source'].apply(
                lambda x: [
                    y.replace(tokenizer.sep_token, '').replace(tokenizer.cls_token, '').replace(tokenizer.pad_token, '')
                    for y in x])
        self.batch_max_length = self.meta_data['batch_max_length'].values
        self.source = self.meta_data['source'].values
        self.parameter = parameter
        self.max_length = max_length
        self.cell_type = self.meta_data['cell_type'].values
        # self.cell_id = self.meta_data['cell_id'].values
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        source = self.source[index]
        cell_type = self.cell_type[index]
        batch_max_len = min(self.batch_max_length[index], self.max_length)

        cell_inputs = self.tokenizer.batch_encode_plus(
            source,
            add_special_tokens=False,
            max_length=self.parameter.cell_max_length,
            # padding="max_length",
            return_attention_mask=False,
            truncation=True,
        )
        seq, seq_mask, target_mask = self.max_length_rule_base(cell_inputs['input_ids'], cell_type, batch_max_len)
        return seq, seq_mask, target_mask

    def __len__(self):
        return len(self.meta_data)

    def max_length_rule_base(self, cell_inputs, cell_type, batch_max_len):
        init_length = [len(x) for x in cell_inputs]
        total_max_length = batch_max_len - len(init_length)
        min_length = total_max_length // len(init_length)
        cell_length = self.search_length(init_length, min_length, total_max_length, len(init_length))
        # print(init_code_length,code_length)

        seq = []
        for i in range(len(cell_length)):
            if cell_type[i] == 0:
                seq.append(self.tokenizer.cls_token_id)
            else:
                seq.append(self.tokenizer.sep_token_id)

            if cell_length[i] > 0:
                seq.extend(cell_inputs[i][:cell_length[i]])

        # print(len(seq),'1111', np.sum(init_length),np.sum(cell_length))
        if len(seq) < batch_max_len:
            seq_mask = [1] * len(seq) + [0] * (batch_max_len - len(seq))
            seq = seq + [self.tokenizer.pad_token_id] * (batch_max_len - len(seq))
        else:
            seq_mask = [1] * batch_max_len
            seq = seq[:batch_max_len]
        seq, seq_mask = np.array(seq, dtype=np.int), np.array(seq_mask, dtype=np.int)
        target_mask = np.where((seq == self.tokenizer.cls_token_id) | (seq == self.tokenizer.sep_token_id), 1, 0)
        return seq, seq_mask, target_mask

    @staticmethod
    def search_length(init_length, min_length, total_max_length, cell_count, step=4, max_search_count=50):
        if np.sum(init_length) <= total_max_length:
            return init_length

        res = [min(init_length[i], min_length) for i in range(cell_count)]
        for s_i in range(max_search_count):
            tmp = [min(init_length[i], res[i] + step) for i in range(cell_count)]
            if np.sum(tmp) < total_max_length:
                res = tmp
            else:
                break
        for s_i in range(cell_count):
            tmp = [i for i in res]
            tmp[s_i] = min(init_length[s_i], res[s_i] + step)
            if np.sum(tmp) < total_max_length:
                res = tmp
            else:
                break
        return res

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import RobertaModel, RobertaConfig, AutoConfig, AutoModel, AutoModelForMaskedLM


class MarkdownModel(nn.Module):
    def __init__(self, name, num_classes=1, seq_length=96, pretrained=True):
        super(MarkdownModel, self).__init__()
        # self.encoder = AutoModel.from_pretrained(name, attention_probs_dropout_prob=0.1, hidden_dropout_prob=0.1)
        self.config = AutoConfig.from_pretrained(name)
        self.config.attention_probs_dropout_prob = 0.
        self.config.hidden_dropout_prob = 0.
        self.config.max_position_embeddings = 4096 * 2 
        # self.config.output_hidden_states = True
        if pretrained:
            self.encoder = AutoModel.from_pretrained(name, config=self.config, ignore_mismatched_sizes=True)
            # self.encoder = AutoModelForMaskedLM.from_pretrained(name, config=self.config)
        else:
            # self.encoder = AutoModelForMaskedLM.from_config(self.config)
            self.encoder = AutoModel.from_config(self.config)

        # self.encoder = AutoModel.from_pretrained(name)
        # print(self.encoder.__dict__)
        # transformer_layers = 2
#         self.seq_length = seq_length
#         self.transformer_layers = transformer_layers
        self.in_dim = self.encoder.config.hidden_size
        print(self.in_dim)
#         self.pe = PositionalEncoding(self.in_dim)
#         self.trans = nn.Sequential(
#             *[TransformerBlock(emb_s=64, head_cnt=self.in_dim // 64, dp1=0., dp2=0.) for _ in
#               range(transformer_layers)])
        self.bilstm = nn.LSTM(self.in_dim, self.in_dim, num_layers=1, 
                              dropout=self.config.hidden_dropout_prob, batch_first=True,
                              bidirectional=True)
        # self.dropouts = nn.ModuleList([nn.Dropout(0.5) for _ in range(5)])
#         hidden = 64
#         dropout = 0.
#         self.sequence = nn.Sequential(
#             # nn.BatchNorm1d(1),
#             nn.Linear(1, hidden),  # todo
#             nn.Dropout(dropout),
#             nn.ReLU(),
#             # nn.BatchNorm1d(hidden),
#             nn.Linear(hidden, hidden),
#             nn.Dropout(dropout),
#             nn.ReLU()
#         )
        self.last_fc = nn.Linear(self.in_dim*2, num_classes)
        # self.fc = nn.LazyLinear(num_classes)
        torch.nn.init.normal_(self.last_fc.weight, std=0.02)
        self.sig = nn.Sigmoid()

    def forward(self, x, mask):
        x = self.encoder(x, attention_mask=mask)["last_hidden_state"]
        # x = x.reshape(-1, code_count, self.seq_length, self.in_dim).mean(2)
        #         x = torch.sum(x * mask.unsqueeze(-1), dim=1) / torch.sum(mask, dim=1).unsqueeze(-1)
        #         x = x.reshape(-1, code_count, self.in_dim)
        # x = x + self.sequence(dense_features.unsqueeze(-1))
        # print(x)
        # print(x.shape)
#         prev = None
#         x = self.pe(x)
#         for i in range(self.transformer_layers):
#             # x = x * mask.unsqueeze(-1)
#             x, prev = self.trans[i](x, prev)
        # x = torch.sum(x * mask.unsqueeze(-1), dim=1) / torch.sum(mask, dim=1).unsqueeze(-1)
        # x = torch.cat([x, self.sequence(dense_features.unsqueeze(1)).repeat(1,2048,1)], dim=2)
        # x = x.mean(1)
        x, _ = self.bilstm(x)
        out = self.last_fc(x)
#         for i, dropout in enumerate(self.dropouts):
#             if i == 0:
#                 out = self.last_fc(dropout(x))
#             else:
#                 out += self.last_fc(dropout(x))
#         out /= len(self.dropouts)
        # out = self.sig(out)
        out = out.squeeze(-1)
        return out
# input = torch.randn(2, 200).long() +10
# input2 = torch.zeros(2, 200)
# net = MarkdownModel('roberta-base', pretrained=False)
# print(input)
# print(net(input, input2))

In [ ]:
%%writefile parameter.py
import torch


class Parameter(object):
    def __init__(self):
        # data
        self.result_dir = './user_data/'
        self.data_dir = '../input/AI4Code/'
        self.k_folds = 5
        self.n_jobs = 4
        self.random_seed = 27
        self.seq_length = 512
        self.cell_count = 128
        self.cell_max_length = 128
        self.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
        # model
        self.use_cuda = torch.cuda.is_available()
        self.gpu = 0
        self.print_freq = 100
        self.lr = 0.003
        self.weight_decay = 0
        self.optim = 'Adam'
        self.base_epoch = 30

    def get(self, name):
        return getattr(self, name)

    def set(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

    def __str__(self):
        return '\n'.join(['%s:%s' % item for item in self.__dict__.items()])


if __name__ == '__main__':
    parameter = Parameter()
    print(parameter)

In [ ]:
%%writefile utils.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
import sys
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
import itertools


class KFold(object):
    """
    KFold: Group split by group_col or random_split
    """

    def __init__(self, random_seed, k_folds=10, flag_name='fold_flag'):
        self.k_folds = k_folds
        self.flag_name = flag_name
        np.random.seed(random_seed)

    def group_split(self, train_df, group_col):
        group_value = list(set(train_df[group_col]))
        group_value.sort()
        fold_flag = [i % self.k_folds for i in range(len(group_value))]
        np.random.shuffle(fold_flag)
        train_df = train_df.merge(pd.DataFrame({group_col: group_value, self.flag_name: fold_flag}), how='left',
                                  on=group_col)
        return train_df

    def random_split(self, train_df):
        fold_flag = [i % self.k_folds for i in range(len(train_df))]
        np.random.shuffle(fold_flag)
        train_df[self.flag_name] = fold_flag
        return train_df

    def stratified_split(self, train_df, group_col):
        train_df[self.flag_name] = 1
        train_df[self.flag_name] = train_df.groupby(by=[group_col])[self.flag_name].rank(ascending=True,
                                                                                         method='first').astype(int)
        train_df[self.flag_name] = train_df[self.flag_name].sample(frac=1.0).reset_index(drop=True)
        train_df[self.flag_name] = train_df[self.flag_name] % self.k_folds
        return train_df


# http://stackoverflow.com/questions/34950201/pycharm-print-end-r-statement-not-working
class Logger(object):
    def __init__(self):
        self.terminal = sys.stdout  # stdout
        self.file = None

    def open(self, file, mode=None):
        if mode is None: mode = 'w'
        self.file = open(file, mode)

    def write(self, message, is_terminal=1, is_file=1):
        if '\r' in message: is_file = 0

        if is_terminal == 1:
            self.terminal.write(message)
            self.terminal.flush()
            # time.sleep(1)

        if is_file == 1:
            self.file.write(message)
            self.file.flush()

    def flush(self):
        # this flush method is needed for python 3 compatibility.
        # this handles the flush command by doing nothing.
        # you might want to specify some extra behavior here.
        pass


def seed_everything(random_seed):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    os.environ["PYTHONHASHSEED"] = str(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)
        #         torch.backends.cudnn.enabled = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.
        self.avg = 0.
        self.sum = 0.
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def save_model(model, save_path, model_name):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    filename = os.path.join(save_path, model_name + '.pth.tar')
    torch.save({'state_dict': model.state_dict(), }, filename)
    # if is_best:
    #     best_filename = os.path.join(save_path, model_name + '_best_model.pth.tar')
    #     shutil.copyfile(filename, best_filename)


def load_model(model, load_path, model_name):
    if not os.path.exists(load_path):
        os.makedirs(load_path)
    filename = os.path.join(load_path, model_name + '.pth.tar')
    model.load_state_dict(torch.load(filename)['state_dict'])
    return model


def adjust_learning_rate(optimizer, epoch, args):
    """Sets the learning rate to the initial LR decayed every 10 epochs"""
    # lr = args.lr * (0.5 ** (epoch // 10))
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['lr'] * (0.3 ** (epoch // 10))


def worker_init_fn(worker_id):
    """
    Handles PyTorch x Numpy seeding issues.

    Args:
        worker_id (int): Id of the worker.
    """
    np.random.seed(np.random.get_state()[1][0] + worker_id)


class MyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs, rank, rank_mask):
        # loss = (inputs - rank) ** 2 * rank_mask
        loss = torch.abs(inputs - rank) * rank_mask
        loss = torch.sum(loss, dim=1) / (torch.sum(rank_mask, dim=1) + 1)
        loss = loss.mean()
        return loss


class MyBCELoss(nn.Module):
    def __init__(self, class_weight=False):
        super().__init__()
        self.class_weight = class_weight

    def forward(self, inputs, targets, mask, sample_weight=None):
        # print(inputs)
        # inputs = inputs[:,:targets.shape[1]]
        bce1 = F.binary_cross_entropy(inputs, torch.ones_like(inputs), reduction='none')
        bce2 = F.binary_cross_entropy(inputs, torch.zeros_like(inputs), reduction='none')
        bce = 1 * bce1 * targets + bce2 * (1 - targets)
        # mask = torch.where(targets >= 0, torch.ones_like(bce), torch.zeros_like(bce))
        bce = bce * mask
        # print(bce)
        #         if sample_weight is not None:
        #             bce = bce * sample_weight.unsqueeze(1)
        loss = bce.mean()  # .sum() / mask.sum()
        return loss


class FGM():
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1., emb_name='emb'):
        # emb_name这个参数要换成你模型中embedding的参数名
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                # print(name, param)
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / max(norm, 0.001)
                    param.data.add_(r_at)

    def restore(self, emb_name='emb'):
        # emb_name这个参数要换成你模型中embedding的参数名
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}


from bisect import bisect


# from https://www.kaggle.com/code/ryanholbrook/competition-metric-kendall-tau-correlation
# Actually O(N^2), but fast in practice for our data
def count_inversions(a):
    inversions = 0
    sorted_so_far = []
    for i, u in enumerate(a):  # O(N)
        j = bisect(sorted_so_far, u)  # O(log N)
        inversions += i - j
        sorted_so_far.insert(j, u)  # O(N)
    return inversions


def kendall_tau(ground_truth, predictions):
    total_inversions = 0  # total inversions in predicted ranks across all instances
    total_2max = 0  # maximum possible inversions across all instances
    for gt, pred in zip(ground_truth, predictions):
        assert len(gt) == len(pred)
        ranks = [gt.index(x) for x in pred]  # rank predicted order in terms of ground truth
        total_inversions += count_inversions(ranks)
        n = len(gt)
        total_2max += n * (n - 1)
    return 1 - 4 * total_inversions / total_2max


def get_score(df, masks, rank_pred, code_df_valid):
    df['cell_id2'] = [[y[i] for i in range(len(x)) if x[i] == 1] for x, y in
                          zip(df['cell_type'].values, df['cell_id'].values)]
    df = df[['id', 'cell_id2']].explode('cell_id2')
    df = df[~pd.isnull(df['cell_id2'])]
    preds = rank_pred.flatten()[np.where(masks.flatten() == 1)]
    df['rank2'] = preds
    df = df.groupby(by=['id', 'cell_id2'], as_index=False)['rank2'].agg('mean')

    df.rename(columns={'cell_id2': 'cell_id'}, inplace=True)
    code_df_valid_tmp = code_df_valid[code_df_valid['id'].isin(df['id'])]
    df = pd.concat([df[['id', 'cell_id', 'rank2']], code_df_valid_tmp]).reset_index(drop=True)
    df = df.sort_values(by=['id', 'rank2'], ascending=True)
    res = df.groupby(by=['id'], sort=False, as_index=False)['cell_id'].agg(list)

    train_orders = pd.read_csv('../input/AI4Code/train_orders.csv')
    train_orders['cell_order'] = train_orders['cell_order'].str.split()
    res = res.merge(train_orders, how='left', on='id')
    print(res)
    score = kendall_tau(res['cell_order'], res['cell_id'])
    return score


# # https://www.kaggle.com/code/anyai28/fast-inference-by-padding-optimization
# def get_sorted_test_df(df, extract_col, tokenizer, batch_size):
#     input_lengths = []
#     for text in df[extract_col].fillna("").values:
#         length = len(tokenizer(text, add_special_tokens=True)['input_ids'])
#         input_lengths.append(length)
#     df['input_lengths'] = input_lengths
#     length_sorted_idx = np.argsort([-l for l in input_lengths])

#     # sort dataframe
#     sort_df = df.iloc[length_sorted_idx]
#     # calc max_len per batch
#     sorted_input_length = sort_df['input_lengths'].values
#     batch_max_length = np.zeros_like(sorted_input_length)
#     for i in range((len(sorted_input_length) // batch_size) + 1):
#         batch_max_length[i * batch_size:(i + 1) * batch_size] = np.max(
#             sorted_input_length[i * batch_size:(i + 1) * batch_size])
#     sort_df['batch_max_length'] = batch_max_length
#     return sort_df, length_sorted_idx

# https://www.kaggle.com/code/anyai28/fast-inference-by-padding-optimization
def get_sorted_test_df(df, extract_col, tokenizer, batch_size, cell_max_length=128):
    input_lengths = []
    for text in df[extract_col].values:
        # print(text)
        tmp = tokenizer.batch_encode_plus(
            text,
            add_special_tokens=False,
            max_length=cell_max_length,
            return_attention_mask=False,
            truncation=True,
        )
        init_length = [len(x) for x in tmp['input_ids']]
        total_length = np.sum(init_length) + len(init_length)
        input_lengths.append(total_length)
    # print(input_lengths)
    df['input_lengths'] = input_lengths
    length_sorted_idx = np.argsort([-l for l in input_lengths])

    # sort dataframe
    sort_df = df.iloc[length_sorted_idx]
    # calc max_len per batch
    sorted_input_length = sort_df['input_lengths'].values
    batch_max_length = np.zeros_like(sorted_input_length)
    total_iter = len(sorted_input_length) // batch_size if len(sorted_input_length) % batch_size == 0 else (len(sorted_input_length) // batch_size) + 1
    for i in range(total_iter):
        batch_max_length[i * batch_size:(i + 1) * batch_size] = np.max(
            sorted_input_length[i * batch_size:(i + 1) * batch_size])
    sort_df['batch_max_length'] = batch_max_length
    return sort_df, length_sorted_idx


def get_model_path(model_name):
    res = '../input/'
    if model_name in ['distilroberta-base', 'roberta-base', 'roberta-large']:
        res += 'roberta-transformers-pytorch/' + model_name
    elif model_name in ['bart-base', 'bart-large']:
        res += 'bartbase' if model_name == 'bart-base' else 'bartlarge'
        res += '/'
    elif model_name in ['deberta-base', 'deberta-large', 'deberta-v2-xlarge', 'deberta-v2-xxlarge']:
        res += 'deberta/' + model_name.replace('deberta-', '')
    elif model_name in ['deberta-v3-large']:
        res += 'deberta-v3-large/' + model_name
    elif model_name in ['electra-base', 'electra-large']:
        res += 'electra/' + model_name + '-discriminator'
    elif 'albert' in model_name:
        res += 'pretrained-albert-pytorch/' + model_name
    elif model_name == 'funnel-large':
        res += 'funnel-large/'
    elif model_name == 'xlnet-base':
        res += 'xlnet-pretrained/xlnet-pretrained/'
    elif model_name == 'deberta-base-mnli':
        res += 'huggingface-deberta-variants/deberta-base-mnli/deberta-base-mnli/'
    elif model_name == 'deberta-xlarge':
        res += 'huggingface-deberta-variants/deberta-xlarge/deberta-xlarge/'
    elif model_name == 'codebert-base':
        res += 'codebert-base/codebert-base/'
    elif model_name == 'CodeBERTa-small-v1':
        res += 'huggingface-code-models/CodeBERTa-small-v1/'
    else:
        raise ValueError(model_name)
    return res

In [ ]:
%%writefile predict.py
# coding=utf-8
import numpy as np
import pandas as pd
import os
import re
import sys
import gc
import time
from transformers import BertTokenizer, RobertaTokenizerFast, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from parameter import Parameter
from models import MarkdownModel
from dataset import MarkdownDataset, MarkdownDatasetV2
from data_processing import read_json_data, preprocess_df, get_truncated_df, get_truncated_df2
from utils import *

parameter = Parameter()
parameter.set(**{'batch_size': 2, 'n_jobs': 2})
seed_everything(parameter.random_seed)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

log_dir = './inference_log.txt'
if os.path.exists(log_dir):
    os.remove(log_dir)
log = Logger()
log.open(log_dir, mode='a')


def predict(model, data_loader, max_length):
    # switch to evaluate mode
    model.eval()
    y_pred = []
    mask = []
    for i, batch_data in enumerate(data_loader):
        if parameter.use_cuda:
            batch_data = (t.cuda() for t in batch_data)
        seq, seq_mask, target_mask = batch_data
        outputs = model(seq, seq_mask).detach().cpu().numpy()
        target_mask = target_mask.detach().cpu().numpy().reshape((outputs.shape[0], -1))
        tmp1 = np.zeros((outputs.shape[0], max_length))
        tmp1[:, :outputs.shape[1]] = outputs
        tmp2 = np.zeros((outputs.shape[0], max_length))
        tmp2[:, :outputs.shape[1]] = target_mask
        
        y_pred.append(tmp1)
        mask.append(tmp2)
        
    y_pred = np.concatenate(y_pred)
    mask = np.concatenate(mask)
    return y_pred, mask


def get_preds(my_df, my_loader, my_model, model_path, max_length=4096):
    if my_df.shape[0] > 0:
        my_model.load_state_dict(torch.load(model_path)['state_dict'])
    if parameter.use_cuda:
        my_model = my_model.cuda()
    with torch.no_grad():
        y_pred, mask = predict(my_model, my_loader, max_length)
    return y_pred, mask

def get_results(df, masks, rank_pred, code_df_valid):
    df['cell_id2'] = df['cell_id']
    df = df[['id', 'cell_id2']].explode('cell_id2')
    df = df[~pd.isnull(df['cell_id2'])]
    preds = rank_pred.flatten()[np.where(masks.flatten() == 1)]
    df['rank2'] = preds
    df = df.groupby(by=['id', 'cell_id2'], as_index=False)['rank2'].agg('mean')

    df.rename(columns={'cell_id2': 'cell_id'}, inplace=True)
    code_df_valid_tmp = code_df_valid[code_df_valid['id'].isin(df['id'])]
    code_df_valid_tmp['rank3'] = code_df_valid_tmp.groupby(by=['id'])['rank2'].rank(ascending=True,method='first')
    tmp = code_df_valid_tmp[['id','cell_id','rank3']].merge(df, how='inner',on=['id', 'cell_id'])
    tmp['rank4'] = tmp.groupby(by=['id'])['rank2'].rank(ascending=True, method='first')
    tmp = tmp[['id','cell_id','rank3']].merge(tmp[['id', 'rank4', 'rank2']].rename(columns={'rank4':'rank3'}), how='inner', on=['id', 'rank3'])
    tmp = tmp[['id', 'cell_id', 'rank2']]
    
    df = df.merge(tmp[['id', 'cell_id','rank2']].rename(columns={'rank2':'rank3'}), how='left', on=['id', 'cell_id'])
    df['rank2'] = np.where(pd.isnull(df['rank3']),df['rank2'], df['rank3'])
    
    # df = pd.concat([df[['id', 'cell_id', 'rank2']], code_df_valid_tmp]).reset_index(drop=True)
    df = df.sort_values(by=['id', 'rank2'], ascending=True)
    return df

def get_results2(df, masks, rank_pred, code_df_valid):
    df = df[['id', 'id2', 'cell_id']].explode('cell_id')
    df = df[~pd.isnull(df['cell_id'])]
    preds = rank_pred.flatten()[np.where(masks.flatten() == 1)]
    df['rank2'] = preds

#     code_df_valid_tmp = code_df_valid[code_df_valid['id'].isin(df['id'])]
#     code_df_valid_tmp['rank3'] = code_df_valid_tmp.groupby(by=['id'])['rank2'].rank(ascending=True,method='first')
#     tmp = code_df_valid_tmp[['id','cell_id','rank3']].merge(df, how='inner',on=['id', 'cell_id'])
#     tmp['rank4'] = tmp.groupby(by=['id'])['rank2'].rank(ascending=True, method='first')
#     tmp = tmp[['id','cell_id','rank3']].merge(tmp[['id', 'rank4', 'rank2']].rename(columns={'rank4':'rank3'}), how='inner', on=['id', 'rank3'])
#     tmp = tmp[['id', 'cell_id', 'rank2']]
    
#     df = df.merge(tmp[['id', 'cell_id','rank2']].rename(columns={'rank2':'rank3'}), how='left', on=['id', 'cell_id'])
#     df['rank2'] = np.where(pd.isnull(df['rank3']),df['rank2'], df['rank3'])
    
    # df = pd.concat([df[['id', 'cell_id', 'rank2']], code_df_valid_tmp]).reset_index(drop=True)
    df = df.sort_values(by=['id','id2','rank2'], ascending=True)
    return df

log.write('>> reading test_df\n')
test_df = read_json_data(mode='test')
test_df['rank'], test_df['fold_flag'] = 1,-1
test_df = preprocess_df(test_df)

test_df = pd.concat(
        [test_df[test_df['cell_type'] == 0], test_df[test_df['cell_type'] == 1].sample(frac=1.0)]).reset_index(
        drop=True)
test_df['rank2'] = (test_df.groupby(by=['id', 'cell_type']).cumcount() + 1) / \
                    test_df.groupby(by=['id', 'cell_type'])['cell_id'].transform('count')
test_df.loc[test_df['cell_type'] == 1, 'rank2'] = -1
code_df_sub = test_df[test_df['cell_type'] == 0][['id', 'cell_id', 'rank2']].copy()

# test_df2 = test_df[test_df['cell_count']>=96]
test_df = get_truncated_df(test_df, cell_count=parameter.cell_count)


log.write('>> predicting...\n')
start = time.time()
# --------------------
model_name = 'deberta-v3-large'
tokenizer_path = get_model_path(model_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
sort_df, length_sorted_idx = get_sorted_test_df(test_df, 'source', tokenizer, batch_size=parameter.batch_size, cell_max_length=parameter.cell_max_length)
del test_df
gc.collect()

# # -------------------- part1
# test_dataset = MarkdownDatasetV2(sort_df, tokenizer, parameter=parameter, max_length=4096)
# test_loader = DataLoader(test_dataset, shuffle=False, batch_size=2,
#                          num_workers=parameter.n_jobs, drop_last=False, pin_memory=True)
# model = MarkdownModel(get_model_path(model_name), pretrained=False)
# model_path = '../input/ai4code-model/deberta-v3-large_fold0.pth.tar'
# y_preds, masks = get_preds(sort_df, test_loader, model, model_path, max_length=4096)
# del model
# gc.collect()
# torch.cuda.empty_cache()

sort_df1 = sort_df[sort_df['batch_max_length'] <= 4096]
sort_df2 = sort_df[sort_df['batch_max_length'] > 4096]

# -------------------- part1
test_dataset = MarkdownDatasetV2(sort_df1, tokenizer, parameter=parameter, max_length=4096)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=2,
                         num_workers=parameter.n_jobs, drop_last=False, pin_memory=True)
model = MarkdownModel(get_model_path(model_name), pretrained=False)
model_path = '../input/ai4code-model/deberta-v3-large_fold0.pth.tar'
y_preds, masks = get_preds(sort_df1, test_loader, model, model_path, max_length=4096+1024)
del model
gc.collect()
torch.cuda.empty_cache()
    
if len(sort_df2) > 0:
    # -------------------- part2
    test_dataset = MarkdownDatasetV2(sort_df2, tokenizer, parameter=parameter, max_length=4096+1024)
    test_loader = DataLoader(test_dataset, shuffle=False, batch_size=1,
                             num_workers=parameter.n_jobs, drop_last=False, pin_memory=True)
    model = MarkdownModel(get_model_path(model_name), pretrained=False)
    model_path = '../input/ai4code-model/deberta-v3-large_fold0.pth.tar'
    y_preds2, masks2 = get_preds(sort_df2, test_loader, model, model_path ,max_length=4096+1024)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    y_preds = np.concatenate([y_preds2, y_preds])
    masks = np.concatenate([masks2, masks])


res = get_results(sort_df, masks, y_preds, code_df_sub)
# res = pd.concat([res1, res2]).reset_index(drop=True)
# sub_df = res1
# sub_df = res1.sort_values(by=['id', 'rank2'], ascending=True)
sub_df = res.groupby(by=['id'], sort=False)['cell_id'].apply(lambda x: ' '.join(x)).reset_index()
sub_df.rename(columns={'cell_id': 'cell_order'}, inplace=True)
# sub_df['cell_order'] = sub_df['cell_order'].apply(lambda x: ' '.join(x.split()[::-1]))
# print(test_df.shape)
sub_df[['id', 'cell_order']].to_csv('submission.csv', index=False)

In [ ]:
!python predict.py